# Architecture Comparison

Compare different model architectures on the same data:
- Transformer (default)
- CNN
- LSTM
- Conv-Transformer (hybrid)
- RWKV

In [ ]:
import random
import os
import json
import subprocess
import sys

# Generate synthetic data
random.seed(42)
os.makedirs('/tmp/arch-compare', exist_ok=True)
os.chdir('/tmp/arch-compare')

with open('sequences.fasta', 'w') as f:
    for i in range(100):
        length = random.randint(200, 500)
        seq = ''.join(random.choice('ATCG') for _ in range(length))
        f.write(f'>seq_{i}\n{seq}\n')

print('Data generated')

## Prepare Data

In [ ]:
!genomic-research init --fasta sequences.fasta --task pretrain --tokenizer char --max-length 128

## Run Benchmark

Use the built-in benchmark command to compare architectures.

In [ ]:
!genomic-research benchmark --models transformer,cnn,lstm --time 30

## Manual Comparison

For more control, run each architecture separately with config overrides.

In [ ]:
architectures = ['transformer', 'cnn', 'lstm']
results = {}

for arch in architectures:
    print(f'\n=== Training {arch} ===')
    
    config = {'model_type': arch, 'd_model': 128, 'n_layers': 4}
    config_path = f'/tmp/arch-compare/_config_{arch}.json'
    with open(config_path, 'w') as f:
        json.dump(config, f)
    
    env = os.environ.copy()
    env['GENOMIC_TIME_BUDGET'] = '30'
    env['GENOMIC_CONFIG'] = config_path
    
    result = subprocess.run(
        [sys.executable, 'train.py'],
        capture_output=True, text=True, env=env, timeout=60
    )
    
    # Parse val_score from output
    for line in result.stdout.split('\n'):
        if line.startswith('val_score:'):
            score = float(line.split(':')[1].strip())
            results[arch] = score
            print(f'{arch}: val_score = {score:.4f}')

print('\n=== Results ===')
for arch, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'{arch:20s} val_score = {score:.4f}')

## Visualize Results

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if results:
    archs = list(results.keys())
    scores = [results[a] for a in archs]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(archs, scores, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'][:len(archs)])
    ax.set_ylabel('val_score (higher is better)')
    ax.set_title('Architecture Comparison')
    
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{score:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('architecture_comparison.png', dpi=150)
    plt.show()
    print('Saved to architecture_comparison.png')